# [EXPLORATION] Netztransparenz API - Activated Balancing Volumes

## Header & Resources

- **Official API Portal:** https://ds.netztransparenz.de/api/v1/data  
- **API Documentation / Auth:** https://identity.netztransparenz.de/users/connect/token  
- **Key Endpoint Parameters:**
  - `series` (e.g., `AktivierteSRL`, `AktivierteMRL`, `NRVSaldo`, `RZSaldo`, `reBAP`)
  - `quality` (e.g., `Qualitaetsgesichert`, `Betrieblich`)
  - `start`, `end` (ISO8601 window, requested in UTC format)

This notebook is structured as a scientific validation document for raw API behavior, data quality, and timezone integrity before ingestion.

## Scientific Notation & Mapping Table

| Original API Column | Thesis Notation (LaTeX) | Unit | Description |
|---|---|---|---|
| AktivierteSRL Positiv | $P^{+}_{aFRR,t}$ | MW | Activated positive aFRR power at time $t$. |
| AktivierteSRL Negativ | $P^{-}_{aFRR,t}$ | MW | Activated negative aFRR power at time $t$. |
| AktivierteMRL Positiv | $P^{+}_{mFRR,t}$ | MW | Activated positive mFRR power at time $t$. |
| AktivierteMRL Negativ | $P^{-}_{mFRR,t}$ | MW | Activated negative mFRR power at time $t$. |
| NRVSaldo | $NRV_t$ | MW | Net regulation volume balance at time $t$. |
| RZSaldo | $RZ_t$ | MW | Control zone saldo at time $t$. |
| reBAP | $P_{reBAP,t}$ | EUR/MWh | Balancing energy settlement price at time $t$. |
| Activated Energy (derived) | $\Delta E_{act,t}$ | MWh | Derived energy from power and resolution, e.g. $\Delta E_{act,t}=P_{act,t}\cdot\Delta t$. |


## Implementation

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta, timezone
import io
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import requests
from dotenv import load_dotenv
from IPython.display import display

for candidate in (Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parents[1] / '.env'):
    if candidate.exists():
        load_dotenv(candidate)

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'api' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from energy_trading.visualization.style import apply_geo_style

apply_geo_style()


In [2]:
def _ensure_bearer(token: str) -> str:
    return token if token.startswith('Bearer ') else f'Bearer {token}'


def _fetch_token_from_client_credentials(client_id: str, client_secret: str) -> str:
    token_url = 'https://identity.netztransparenz.de/users/connect/token'
    payload = {
        'grant_type': 'client_credentials',
        'scope': 'api',
        'client_id': client_id,
        'client_secret': client_secret,
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/x-www-form-urlencoded',
    }

    resp = requests.post(token_url, headers=headers, data=payload, timeout=30)
    resp.raise_for_status()
    access_token = resp.json().get('access_token')
    if not access_token:
        raise RuntimeError('Token response does not contain access_token')
    return _ensure_bearer(access_token)


def get_token() -> str:
    token = os.getenv('NETZTRANSPARENZ_TOKEN')
    if token:
        return _ensure_bearer(token)

    client_id = os.getenv('NETZTRANSPARENZ_CLIENT_ID')
    client_secret = os.getenv('NETZTRANSPARENZ_CLIENT_SECRET')
    if client_id and client_secret:
        return _fetch_token_from_client_credentials(client_id, client_secret)

    raise RuntimeError(
        'No credentials configured. Set NETZTRANSPARENZ_TOKEN or '
        'NETZTRANSPARENZ_CLIENT_ID/NETZTRANSPARENZ_CLIENT_SECRET.'
    )


def _fmt_utc(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime('%Y-%m-%dT%H:%MZ')


def fetch_netztransparenz_csv(
    series: str,
    quality: str,
    start_utc: datetime,
    end_utc: datetime,
    token: str,
    timeout: int = 60,
) -> pd.DataFrame:
    base_url = 'https://ds.netztransparenz.de/api/v1/data'
    url = f"{base_url}/NrvSaldo/{series}/{quality}/{_fmt_utc(start_utc)}/{_fmt_utc(end_utc)}"
    headers = {'Authorization': token, 'Accept': 'text/csv'}

    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f'Netztransparenz request failed for {series}: {exc}') from exc

    try:
        df = pd.read_csv(io.StringIO(resp.text), sep=';', decimal=',', low_memory=False)
    except Exception as exc:
        raise RuntimeError(f'CSV parsing failed for {series}: {exc}') from exc

    df.columns = [str(c).strip() for c in df.columns]
    return df


START_UTC = datetime(2025, 8, 1, 10, 0, tzinfo=timezone.utc)
END_UTC = START_UTC + timedelta(days=7)

TOKEN = get_token()
df_raw = fetch_netztransparenz_csv(
    series='AktivierteSRL',
    quality='Qualitaetsgesichert',
    start_utc=START_UTC,
    end_utc=END_UTC,
    token=TOKEN,
)

print(f'Raw rows: {len(df_raw)}, columns: {len(df_raw.columns)}')
print('First 5 rows (raw response):')
display(df_raw.head(5))


HTTPError: 400 Client Error: Bad Request for url: https://identity.netztransparenz.de/users/connect/token

## Inspection & Data Integrity

In [ ]:
print('--- df.info() ---')
df_raw.info()

na_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
na_pct_nonzero = na_pct[na_pct > 0]

fig, ax = plt.subplots(figsize=(12, 5))
if len(na_pct_nonzero) == 0:
    ax.text(0.5, 0.5, 'No missing values detected', ha='center', va='center')
    ax.set_axis_off()
else:
    na_pct_nonzero.plot(kind='bar', ax=ax, color='#226E9C')
    ax.set_ylabel('Missing values [%]')
    ax.set_title('Missingness by column (percentage)')
    ax.tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()


In [ ]:
df_work = df_raw.copy()

date_col = next((c for c in df_work.columns if c.lower() == 'datum'), None)
time_col = next((c for c in df_work.columns if c.lower() in {'von', 'time'}), None)

if date_col and time_col:
    ts_local = pd.to_datetime(
        df_work[date_col].astype(str) + ' ' + df_work[time_col].astype(str),
        format='%d.%m.%Y %H:%M',
        errors='coerce',
    )
    ts_local = ts_local.dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward')
    df_work = df_work.assign(timestamp_local=ts_local, timestamp_utc=ts_local.dt.tz_convert('UTC'))

    pos_col = next((c for c in df_work.columns if 'positiv' in c.lower()), None)
    if pos_col is not None:
        y = pd.to_numeric(df_work[pos_col], errors='coerce')
        plot_df = pd.DataFrame({'timestamp_utc': df_work['timestamp_utc'], 'value': y}).dropna().sort_values('timestamp_utc')

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(plot_df['timestamp_utc'], plot_df['value'], color='#226E9C', linewidth=1.8)
        ax.set_title('Sanity Plot: AktivierteSRL Positiv')
        ax.set_xlabel('Timestamp (UTC)')
        ax.set_ylabel('MW')
        plt.tight_layout()
        plt.show()

    idx = df_work.set_index('timestamp_utc').index
    print('--- Timezone validation ---')
    print('First timestamp (UTC):', idx.min())
    print('Last timestamp  (UTC):', idx.max())
    print('Detected df.index.tz:', idx.tz)

    tz_col = next((c for c in df_raw.columns if c.lower() == 'zeitzone'), None)
    if tz_col is not None:
        print('Raw Zeitzone values (top):')
        print(df_raw[tz_col].astype(str).value_counts(dropna=False).head(10))

    print('Interpretation: Local date/time fields are parsed as Europe/Berlin and converted to UTC for merge consistency.')
else:
    print('Could not find date/time columns (Datum + von/time). Timezone validation skipped.')


## Export Strategy

- The validated and normalized API output is persisted in `data/raw/netztransparenz.parquet`.
- Production ingestion logic is implemented in `src/energy_trading/ingestion/fetch_netztransparenz.py`.
- The merge step aligns all sources on `timestamp_utc` to support reproducible multi-source joins in `data/processed/all_data.parquet`.
- Derived modeling datasets are generated downstream from this raw layer to preserve provenance and auditability.
